[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/39_ml_systems/90_exercises.ipynb)

# Bài tập — Chương 39: ML Systems (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/39_ml_systems

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_customer_churn

FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]


def make_model(estimator):
    pre = ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(pre, estimator)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Rò rỉ thời điểm (point-in-time leakage) là gì? (b) Lệch train/serving là gì? (c) Phân biệt lặp lại / tái lập / tái hiện. (d) Lời nguyền người thắng. (e) Vì sao không được `pickle.load` file không tin cậy? (f) Alias trong model registry dùng để làm gì?

✍️ Trả lời:

**Đáp án:**
(a) Đặc trưng của một mẫu huấn luyện dùng thông tin **sau** thời điểm dự đoán (ví dụ tính trên toàn bộ lịch sử đến hôm nay) — offline đẹp, production sụp vì lúc dự đoán thông tin đó chưa tồn tại.
(b) Đặc trưng/tiền xử lý lúc phục vụ khác lúc huấn luyện (code viết lại, nguồn dữ liệu khác, cách điền thiếu khác) → cùng mô hình cho dự đoán khác.
(c) Lặp lại: chạy lại trên máy mình ra cùng kết quả. Tái lập: người khác/máy khác từ code + dữ liệu + cấu hình ra cùng kết quả. Tái hiện: dữ liệu mới, kết luận vẫn đúng.
(d) Chọn cấu hình tốt nhất trong nhiều cấu hình theo điểm validation → điểm của người thắng lạc quan (một phần là may), càng nặng khi validation nhỏ và thử nhiều cấu hình.
(e) Pickle có thể gọi hàm tuỳ ý khi nạp (`__reduce__`) → thực thi code độc hại.
(f) Con trỏ có tên (`champion`, `challenger`) tới một phiên bản; hệ thống phục vụ nạp theo alias → đổi mô hình hay rollback chỉ là đổi con trỏ.

## L1 — Giải thích bằng lời

**Câu 2.** Một nhóm báo cáo: "Sau 200 thí nghiệm, cấu hình mới đạt AUC test 0.842, hơn mô hình đang chạy (0.838). Đề xuất triển khai ngay." Họ dùng một lần chia train/test, seed 42, và chọn cấu hình theo điểm **test**. Hãy chỉ ra các vấn đề và đề xuất quy trình đánh giá đúng.

✍️ Trả lời:

**Gợi ý:** (1) **Chọn trên test** 200 lần → test đã thành validation, 0.842 bị lạc quan (lời nguyền người thắng); cần một holdout chưa từng dùng để chọn. (2) Chênh lệch 0.004 có thể **nhỏ hơn nhiễu** do seed/lần chia — cần lặp nhiều lần chia/fold, so sánh **theo cặp** và khoảng tin cậy. (3) Một lần chia ngẫu nhiên có thể không phản ánh triển khai — dữ liệu có thời gian thì đánh giá theo **thời gian**. (4) Ngoài AUC tổng: hiệu năng **theo nhóm**, độ trễ, kích thước, schema, chi phí vận hành — qua cổng thăng hạng; sau đó shadow/A/B test (chương 40) trước khi thay hoàn toàn. (5) Ghi lại đủ 200 run (tracker) để người khác đánh giá và tái lập.

## L2 — Tính tay

**Câu 3.** Nhật ký hoạt động một khách (ngày 0 → 9): `[1, 1, 0, 1, 0, 0, 1, 0, 0, 0]`. Đặc trưng: `active_last_3` = số ngày hoạt động trong 3 ngày **trước** `as_of`; `days_since_last` = `as_of − 1 − (ngày hoạt động gần nhất trước as_of)`. Tính cho `as_of = 7` (đúng thời điểm) → `correct_last3`, `correct_since`; và cho `as_of = 10` (rò rỉ — dùng dữ liệu tới hôm nay) → `leaky_last3`, `leaky_since`.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
correct_last3, correct_since = 1, 0
leaky_last3, leaky_since = 0, 3
### END CODE HERE ###

In [ ]:
log3 = np.array([1, 1, 0, 1, 0, 0, 1, 0, 0, 0])
check_equal([correct_last3, correct_since], [int(log3[4:7].sum()), 0], "đúng thời điểm (ngày 4–6)")
check_equal([leaky_last3, leaky_since], [int(log3[7:10].sum()), 9 - 6], "rò rỉ (ngày 7–9): thấy khách đã ngừng hoạt động")

**Câu 4.** AUC của hai mô hình trên **cùng** 5 fold: A = `[0.81, 0.79, 0.83, 0.80, 0.82]`, B = `[0.80, 0.79, 0.82, 0.79, 0.80]`. Tính độ lệch chuẩn (ddof = 1) của A → `sd_a`; chênh lệch theo cặp d = A − B: trung bình `mean_diff`, độ lệch chuẩn `sd_diff` (ddof = 1), thống kê $t = \bar d / (s_d/\sqrt 5)$ → `t_stat`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
sd_a = np.sqrt((0 + 0.02 ** 2 + 0.02 ** 2 + 0.01 ** 2 + 0.01 ** 2) / 4)
mean_diff = (0.01 + 0 + 0.01 + 0.01 + 0.02) / 5
sd_diff = np.sqrt((0 + 0.01 ** 2 + 0 + 0 + 0.01 ** 2) / 4)
t_stat = mean_diff / (sd_diff / np.sqrt(5))
### END CODE HERE ###
print(f"sd(A) = {sd_a:.4f} | d̄ = {mean_diff:.4f}, s_d = {sd_diff:.4f}, t = {t_stat:.2f}")

In [ ]:
a4, b4 = np.array([0.81, 0.79, 0.83, 0.80, 0.82]), np.array([0.80, 0.79, 0.82, 0.79, 0.80])
check_close(sd_a, a4.std(ddof=1), message="độ lệch chuẩn của A")
check_close([mean_diff, sd_diff], [(a4 - b4).mean(), (a4 - b4).std(ddof=1)], message="chênh lệch theo cặp")
check_close(t_stat, np.sqrt(10), tol=1e-6, message="t = √10 ≈ 3.16")

**Nhận xét:** Độ lệch chuẩn của A giữa các fold (0.016) lớn hơn chênh lệch trung bình (0.01) — so hai trung bình độc lập thì không phân biệt được. Nhưng B luôn thấp hơn hoặc bằng A **trên cùng fold**, nên chênh lệch theo cặp rất ổn định (s_d = 0.007) → t ≈ 3.16 (p ≈ 0.03 với 4 bậc tự do). Fold khó/dễ ảnh hưởng cả hai mô hình — so sánh theo cặp loại bỏ nhiễu đó. (Thận trọng: các fold dùng chung dữ liệu nên t-test này hơi quá tự tin.)

## L3 — Code

**Bài 5.** Viết `fingerprint(df, row_order_matters=True)`: SHA-256 (16 ký tự hex đầu) của nội dung DataFrame — **không** phụ thuộc thứ tự cột; phụ thuộc thứ tự dòng khi `row_order_matters=True`. Gợi ý: `df.sort_index(axis=1)`, `pd.util.hash_pandas_object(df, index=False)`, `np.sort`, `hashlib.sha256`.

In [ ]:
def fingerprint(df: pd.DataFrame, row_order_matters: bool = True) -> str:
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: return ""
    df = df.sort_index(axis=1)
    row_hashes = pd.util.hash_pandas_object(df, index=False).values
    if not row_order_matters:
        row_hashes = np.sort(row_hashes)
    h = hashlib.sha256()
    h.update("|".join(f"{c}:{t}" for c, t in df.dtypes.items()).encode())
    h.update(row_hashes.tobytes())
    return h.hexdigest()[:16]
    ### END CODE HERE ###


d5 = make_customer_churn(500, seed=5)
print(fingerprint(d5))

In [ ]:
check_equal(len(fingerprint(d5)), 16, "16 ký tự hex")
check_equal(fingerprint(d5), fingerprint(d5.copy()), "cùng dữ liệu → cùng hash")
check_equal(fingerprint(d5), fingerprint(d5[d5.columns[::-1]]), "không phụ thuộc thứ tự cột")
edited5 = d5.copy()
edited5.loc[7, "support_calls"] += 1
check_true(fingerprint(edited5) != fingerprint(d5), "sửa một ô → hash đổi")
shuffled5 = d5.sample(frac=1, random_state=0)
check_true(fingerprint(shuffled5) != fingerprint(d5), "thứ tự dòng có ý nghĩa (mặc định)")
check_equal(fingerprint(shuffled5, row_order_matters=False), fingerprint(d5, row_order_matters=False), "bỏ qua thứ tự dòng")
check_true(fingerprint(d5.astype({"tenure_months": float})) != fingerprint(d5), "đổi kiểu dữ liệu → hash đổi")

**Bài 6.** Viết experiment tracker tối giản `JsonTracker(root)`: `log(params, metrics, tags=None)` ghi một file `run-001.json`, `run-002.json`, … (đánh số tăng dần) chứa `run_id`, `params`, `metrics`, `tags`; trả về `run_id`. `to_frame()` trả DataFrame với cột `run_id`, `params.<tên>`, `metrics.<tên>` — mỗi run một dòng, theo thứ tự `run_id`. Sau đó log 4 run hồi quy logistic với `C ∈ {0.001, 0.01, 0.1, 1}` (metric `val_auc`) và tìm `best_C`.

In [ ]:
class JsonTracker:
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    ### START CODE HERE ### (≈ 14 dòng)
    # STARTER: def log(self, params, metrics, tags=None): ...
    def log(self, params, metrics, tags=None):
        run_id = f"run-{len(list(self.root.glob('run-*.json'))) + 1:03d}"
        record = {"run_id": run_id, "params": params, "metrics": {k: float(v) for k, v in metrics.items()}, "tags": tags or {}}
        (self.root / f"{run_id}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        return run_id

    def to_frame(self):
        rows = []
        for path in sorted(self.root.glob("run-*.json")):
            r = json.loads(path.read_text(encoding="utf-8"))
            rows.append({"run_id": r["run_id"], **{f"params.{k}": v for k, v in r["params"].items()},
                         **{f"metrics.{k}": v for k, v in r["metrics"].items()}})
        return pd.DataFrame(rows)
    ### END CODE HERE ###


d6 = make_customer_churn(3000, seed=6)
X6_tr, X6_va, y6_tr, y6_va = train_test_split(d6[FEATURES], d6.churn, test_size=0.3, random_state=0, stratify=d6.churn)
tracker6 = JsonTracker(Path(tempfile.mkdtemp()) / "runs")
for C in [0.001, 0.01, 0.1, 1.0]:
    m6 = make_model(LogisticRegression(C=C, max_iter=1000)).fit(X6_tr, y6_tr)
    tracker6.log({"C": C}, {"val_auc": roc_auc_score(y6_va, m6.predict_proba(X6_va)[:, 1])}, tags={"data": fingerprint(d6)})
runs6 = tracker6.to_frame()
### START CODE HERE ### (≈ 1 dòng)
best_C = runs6.loc[runs6["metrics.val_auc"].idxmax(), "params.C"]
### END CODE HERE ###
print(runs6.round(4).to_string(index=False), "\nbest C =", best_C)

In [ ]:
check_equal(sorted(p.name for p in tracker6.root.glob("*.json")), ["run-001.json", "run-002.json", "run-003.json", "run-004.json"], "4 file run")
check_equal(list(runs6.columns), ["run_id", "params.C", "metrics.val_auc"], "cột của to_frame")
check_equal(list(runs6["params.C"]), [0.001, 0.01, 0.1, 1.0], "thứ tự run")
check_equal(best_C, runs6.sort_values("metrics.val_auc").iloc[-1]["params.C"], "chọn C có val_auc cao nhất")
check_equal(json.loads((tracker6.root / "run-002.json").read_text(encoding="utf-8"))["tags"]["data"], fingerprint(d6), "tag hash dữ liệu được lưu")

**Bài 7.** Kiểm tra đầu vào không cần thư viện. Viết `infer_schema(df)` → dict theo cột: cột số `{"kind": "number", "min", "max"}`, cột chữ/bool `{"kind": "category", "values": [...]}` (chuỗi, đã sắp xếp); và `validate_record(record, schema)` → **danh sách lỗi** (chuỗi) với các loại: thiếu trường, trường lạ, giá trị thiếu (`None`/NaN), danh mục lạ, số ngoài `[min, max]`. Bản ghi hợp lệ → `[]`.

In [ ]:
def infer_schema(df: pd.DataFrame) -> dict:
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: return {}
    schema = {}
    for col in df.columns:
        if df[col].dtype == bool or df[col].dtype == object:
            schema[col] = {"kind": "category", "values": sorted(map(str, df[col].unique()))}
        else:
            schema[col] = {"kind": "number", "min": float(df[col].min()), "max": float(df[col].max())}
    return schema
    ### END CODE HERE ###


def validate_record(record: dict, schema: dict) -> list:
    ### START CODE HERE ### (≈ 14 dòng)
    # STARTER: return []
    errors = [f"thiếu trường {c}" for c in schema if c not in record]
    errors += [f"trường lạ {c}" for c in record if c not in schema]
    for col, spec in schema.items():
        if col not in record:
            continue
        value = record[col]
        if value is None or (isinstance(value, float) and np.isnan(value)):
            errors.append(f"{col}: giá trị thiếu")
        elif spec["kind"] == "category":
            if str(value) not in spec["values"]:
                errors.append(f"{col}: danh mục lạ {value!r}")
        elif not spec["min"] <= float(value) <= spec["max"]:
            errors.append(f"{col}: {value} ngoài [{spec['min']}, {spec['max']}]")
    return errors
    ### END CODE HERE ###


d7 = make_customer_churn(3000, seed=7)
schema7 = infer_schema(d7[FEATURES])
ok7 = d7[FEATURES].iloc[0].to_dict()
for name, rec in {"hợp lệ": ok7, "hợp đồng lạ": {**ok7, "contract": "weekly"}, "cước âm + thiếu": {**ok7, "monthly_charges": -5.0, "total_charges": None},
                  "sai tên trường": {**{k: v for k, v in ok7.items() if k != "senior"}, "is_senior": True}}.items():
    print(f"{name:16s}: {validate_record(rec, schema7)}")

In [ ]:
check_equal(schema7["contract"], {"kind": "category", "values": ["month-to-month", "one_year", "two_year"]}, "schema cột danh mục")
check_equal(schema7["senior"]["values"], ["False", "True"], "cột bool là danh mục")
check_equal(schema7["tenure_months"]["kind"], "number", "cột số")
check_equal(validate_record(ok7, schema7), [], "bản ghi hợp lệ không có lỗi")
check_equal(len(validate_record({**ok7, "contract": "weekly"}, schema7)), 1, "danh mục lạ → 1 lỗi")
check_equal(len(validate_record({**ok7, "monthly_charges": -5.0, "total_charges": None}, schema7)), 2, "số ngoài khoảng + giá trị thiếu → 2 lỗi")
check_equal(len(validate_record({**ok7, "total_charges": float("nan")}, schema7)), 1, "NaN là giá trị thiếu")
check_equal(len(validate_record({**{k: v for k, v in ok7.items() if k != "senior"}, "is_senior": True}, schema7)), 2, "thiếu trường + trường lạ")

## L4 — Debug

**Bài 8.** Mô hình được huấn luyện trên mảng numpy với cột `log_charges = log1p(monthly_charges)` tính **ngoài** pipeline. Đội backend viết `predict_one(record)` cho API, nhận một dict JSON (thứ tự khoá tuỳ ý). Kiểm thử "parity" (dự đoán online phải khớp dự đoán batch) thất bại. Tìm và sửa (hai lỗi).

In [ ]:
d8 = make_customer_churn(4000, seed=8)
TRAIN_COLS = ["tenure_months", "log_charges", "support_calls", "n_products"]
train8 = d8.iloc[:3000].assign(log_charges=np.log1p(d8.monthly_charges.iloc[:3000]))
model8 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(train8[TRAIN_COLS].values, train8.churn)


def batch_predict(df):
    return model8.predict_proba(df.assign(log_charges=np.log1p(df.monthly_charges))[TRAIN_COLS].values)[:, 1]


def predict_one(record: dict) -> float:
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: x = np.array([[record[k] for k in sorted(record)]])
    # STARTER: return float(model8.predict_proba(x)[0, 1])
    features = {**record, "log_charges": np.log1p(record["monthly_charges"])}
    return float(model8.predict_proba(np.array([[features[c] for c in TRAIN_COLS]]))[0, 1])
    ### END CODE HERE ###


test8 = d8.iloc[3000:]
requests8 = [{"n_products": int(r.n_products), "support_calls": int(r.support_calls), "monthly_charges": float(r.monthly_charges),
              "tenure_months": int(r.tenure_months)} for r in test8.itertuples()]   # thứ tự khoá như JSON từ client
online8 = np.array([predict_one(r) for r in requests8])
print("sai khác lớn nhất online vs batch:", np.abs(online8 - batch_predict(test8)).max())

In [ ]:
check_true(np.allclose(online8, batch_predict(test8)), "parity: online khớp batch")
check_close(predict_one(dict(reversed(list(requests8[0].items())))), online8[0], message="không phụ thuộc thứ tự khoá")

**Lỗi:** (1) `sorted(record)` sắp **tên khoá theo bảng chữ cái** (`monthly_charges, n_products, support_calls, tenure_months`) — khác thứ tự cột lúc huấn luyện; mô hình numpy không có tên cột nên **không báo lỗi**, chỉ dự đoán sai. (2) Thiếu bước `log1p` — tiền xử lý tính ngoài pipeline nên phía phục vụ phải tự viết lại, và đã quên (training–serving skew). Sửa: dựng vector theo **danh sách cột cố định** `TRAIN_COLS` và áp cùng biến đổi. Sửa tận gốc: huấn luyện trên **DataFrame** (sklearn kiểm tra tên cột), đưa `log1p` vào pipeline (`FunctionTransformer` hoặc `ColumnTransformer`) và giữ **kiểm thử parity** này trong CI (chương 40).

## L5 — Phân tích

**Bài 9.** So sánh hồi quy logistic và HGB (mặc định) trên `make_customer_churn(3000, seed=9)` qua **10 lần chia** train/test (seed 0 → 9, test 30%, stratify). Lưu `auc9` (DataFrame cột `LR`, `HGB`), chênh lệch theo cặp `diff9 = LR − HGB`, KTC 95% (phân phối t) của trung bình chênh lệch → `ci9 = (lo, hi)`. Nhận xét: khi nào kết luận được?

In [ ]:
d9 = make_customer_churn(3000, seed=9)
### START CODE HERE ### (≈ 9 dòng)
# STARTER: auc9, diff9, ci9 = None, None, None
rows9 = []
for seed in range(10):
    a, b, ya, yb = train_test_split(d9[FEATURES], d9.churn, test_size=0.3, random_state=seed, stratify=d9.churn)
    rows9.append({"LR": roc_auc_score(yb, make_model(LogisticRegression(max_iter=1000)).fit(a, ya).predict_proba(b)[:, 1]),
                  "HGB": roc_auc_score(yb, make_model(HistGradientBoostingClassifier(random_state=0)).fit(a, ya).predict_proba(b)[:, 1])})
auc9 = pd.DataFrame(rows9)
diff9 = auc9.LR - auc9.HGB
ci9 = stats.t.interval(0.95, len(diff9) - 1, loc=diff9.mean(), scale=diff9.std(ddof=1) / np.sqrt(len(diff9)))
### END CODE HERE ###
print(auc9.agg(["mean", "std"]).round(4))
print(f"LR − HGB = {diff9.mean():+.4f}, s_d = {diff9.std():.4f}, KTC 95% [{ci9[0]:+.4f}, {ci9[1]:+.4f}] | LR thắng {(diff9 > 0).sum()}/10")

In [ ]:
check_equal(auc9.shape, (10, 2), "10 lần chia × 2 mô hình")
check_close(diff9.values, (auc9.LR - auc9.HGB).values, message="chênh lệch theo cặp")
check_less(diff9.std(), auc9.LR.std(), "độ lệch chuẩn theo cặp nhỏ hơn độ lệch chuẩn giữa các lần chia")
check_true(ci9[0] < diff9.mean() < ci9[1], "KTC bao quanh trung bình")
check_greater(ci9[0], 0, "KTC không chứa 0: LR tốt hơn một cách nhất quán")

**Nhận xét:** Trên dữ liệu này quan hệ gần tuyến tính trong log-odds (dữ liệu mô phỏng bằng mô hình logistic) nên hồi quy logistic thắng HGB ở mọi lần chia; KTC của chênh lệch theo cặp nằm hẳn trên 0 → kết luận được. Độ lệch chuẩn của chênh lệch theo cặp nhỏ hơn độ lệch chuẩn AUC giữa các lần chia vì độ khó của mỗi tập test tác động cả hai mô hình. Kết luận được khi chênh lệch **lớn so với nhiễu theo cặp** (KTC không chứa 0) **và** đủ lớn để có ý nghĩa thực tế — còn với HGB và RF ở bài 02 (mục 3), KTC chứa 0 → chọn mô hình đơn giản/rẻ hơn. Bài học hệ thống: mô hình phức tạp hơn **không mặc nhiên** tốt hơn — luôn có baseline đơn giản trong bảng so sánh.

## L6 — Thực tế

**Bài 10.** Viết **cổng thăng hạng** `promotion_gate(candidate, champion, X_hold, y_hold, segment_col="contract", min_gain=0.0, max_segment_drop=0.02)` → `(passed, checks)`, với `checks` là dict `{tên kiểm tra: bool}` gồm: `"overall"` — AUC ứng viên ≥ AUC champion + `min_gain`; `"segments"` — trong mọi nhóm của `segment_col`, AUC ứng viên không thấp hơn champion quá `max_segment_drop`; `"schema"` — `feature_names_in_` của hai mô hình giống nhau. `passed` = mọi kiểm tra đều đúng.

In [ ]:
d10 = make_customer_churn(6000, seed=10)
X10_tr, X10_ho, y10_tr, y10_ho = train_test_split(d10[FEATURES], d10.churn.values, test_size=0.3, random_state=0, stratify=d10.churn)
champion10 = make_model(HistGradientBoostingClassifier(random_state=0)).fit(X10_tr, y10_tr)
good10 = make_model(LogisticRegression(max_iter=1000)).fit(X10_tr, y10_tr)
broken10 = make_model(LogisticRegression(max_iter=1000)).fit(X10_tr.assign(support_calls=0), y10_tr)   # lỗi pipeline dữ liệu


def promotion_gate(candidate, champion, X_hold, y_hold, segment_col="contract", min_gain=0.0, max_segment_drop=0.02):
    ### START CODE HERE ### (≈ 12 dòng)
    # STARTER: return False, {}
    def auc(model, mask):
        return roc_auc_score(y_hold[mask], model.predict_proba(X_hold[mask])[:, 1])

    everyone = np.ones(len(X_hold), dtype=bool)
    drops = [auc(champion, m) - auc(candidate, m)
             for m in [(X_hold[segment_col] == v).values for v in X_hold[segment_col].unique()]]
    checks = {"overall": auc(candidate, everyone) >= auc(champion, everyone) + min_gain,
              "segments": max(drops) <= max_segment_drop,
              "schema": list(candidate.feature_names_in_) == list(champion.feature_names_in_)}
    return all(checks.values()), checks
    ### END CODE HERE ###


print("LR tốt vs champion HGB          →", promotion_gate(good10, champion10, X10_ho, y10_ho))
print("LR dữ liệu lỗi vs champion LR tốt →", promotion_gate(broken10, good10, X10_ho, y10_ho))   # sau khi LR tốt đã thăng hạng
print("LR tốt vs HGB, yêu cầu +0.05     →", promotion_gate(good10, champion10, X10_ho, y10_ho, min_gain=0.05))

In [ ]:
passed_good, checks_good = promotion_gate(good10, champion10, X10_ho, y10_ho)
passed_broken, checks_broken = promotion_gate(broken10, good10, X10_ho, y10_ho)
check_equal(sorted(checks_good), ["overall", "schema", "segments"], "ba kiểm tra")
check_true(passed_good, "mô hình tốt hơn được thăng hạng")
check_true(not passed_broken and not checks_broken["overall"], "mô hình huấn luyện trên dữ liệu lỗi bị chặn")
check_true(not promotion_gate(good10, champion10, X10_ho, y10_ho, min_gain=0.05)[0], "yêu cầu cải thiện tối thiểu quá cao → chặn")
check_true(not promotion_gate(good10, champion10, X10_ho, y10_ho, max_segment_drop=-1)[1]["segments"], "ngưỡng nhóm hoạt động")
narrow10 = make_model(LogisticRegression(max_iter=1000)).fit(X10_tr.drop(columns=["senior"]).assign(senior=False), y10_tr)
check_true(not promotion_gate(narrow10, good10, X10_ho, y10_ho)[1]["schema"], "khác thứ tự/tập cột → schema sai")

**Nhận xét:** Cổng thăng hạng biến "mô hình mới có tốt hơn không?" thành **kiểm tra tự động, lặp lại được** trên holdout cố định. Kiểm tra theo nhóm bắt được trường hợp AUC tổng tăng nhưng một nhóm khách quan trọng bị tệ đi; kiểm tra schema bắt được mô hình train với tập cột khác (API sẽ gửi sai thứ tự/thiếu cột). Trong thực tế thêm: độ trễ, kích thước, calibration (chương 37), công bằng (chương 36), và so với **cả** mô hình baseline đơn giản. Holdout dùng cho cổng phải được làm mới định kỳ (dữ liệu trôi) và không dùng để tinh chỉnh mô hình — nếu không sẽ lại gặp lời nguyền người thắng.

## L7 — Nâng cao

**Bài 11.** **Tập huấn luyện đúng thời điểm, nhiều ngày chụp.** Dùng mô phỏng nhật ký hoạt động của bài 01. Viết `build_snapshot(activity, churn_day, as_of, horizon=30)` → `(X, y)`: chỉ khách còn hoạt động tại `as_of` (`churn_day > as_of`), đặc trưng **chỉ dùng cột ngày `< as_of`** (`active_last_7`, `active_last_30`, `active_total`, `days_since_last`, `trend` — như bài 01), nhãn = rời bỏ trong `(as_of, as_of + horizon]`. Ghép các ngày chụp **60, 75, 90** làm tập train, ngày chụp **120** làm tập test (đánh giá theo thời gian) → `auc11`. Viết thêm **kiểm thử tính chất**: thay toàn bộ dữ liệu từ ngày `as_of` trở đi bằng giá trị ngẫu nhiên thì đặc trưng không đổi → `no_future_dependency` (bool).

In [ ]:
def simulate_activity(n_customers=4000, n_days=180, seed=0):
    rng = np.random.default_rng(seed)
    engagement = rng.beta(2, 3, n_customers)
    churn_day = np.where(rng.random(n_customers) < 0.35, rng.uniform(60, 180, n_customers), np.inf)
    days = np.arange(n_days)
    fading = np.clip((churn_day[:, None] - days[None, :]) / 60, 0, 1)
    p_active = engagement[:, None] * np.where(np.isinf(churn_day)[:, None], 1.0, fading)
    return (rng.random((n_customers, n_days)) < p_active).astype(int), churn_day


activity11, churn11 = simulate_activity(seed=11)


def build_snapshot(activity, churn_day, as_of, horizon=30):
    ### START CODE HERE ### (≈ 10 dòng)
    # STARTER: return None, None
    alive = churn_day > as_of
    hist = activity[alive, :as_of]
    last_active = np.array([np.max(np.flatnonzero(row)) if row.any() else -1 for row in hist])
    X = pd.DataFrame({"active_last_7": hist[:, -7:].sum(1), "active_last_30": hist[:, -30:].sum(1),
                      "active_total": hist.sum(1), "days_since_last": as_of - 1 - last_active,
                      "trend": hist[:, -30:].mean(1) - hist[:, -60:-30].mean(1)})
    y = (churn_day[alive] <= as_of + horizon).astype(int)
    return X, y
    ### END CODE HERE ###


### START CODE HERE ### (≈ 8 dòng)
# STARTER: X11_tr, y11_tr = None, None   # ghép các ngày chụp 60, 75, 90
# STARTER: X11_te, y11_te = None, None   # ngày chụp 120
# STARTER: auc11, no_future_dependency = None, None
parts = [build_snapshot(activity11, churn11, s) for s in [60, 75, 90]]
X11_tr, y11_tr = pd.concat([p[0] for p in parts], ignore_index=True), np.concatenate([p[1] for p in parts])
X11_te, y11_te = build_snapshot(activity11, churn11, 120)
model11 = HistGradientBoostingClassifier(random_state=0).fit(X11_tr, y11_tr)
auc11 = roc_auc_score(y11_te, model11.predict_proba(X11_te)[:, 1])
scrambled = activity11.copy()
scrambled[:, 120:] = np.random.default_rng(0).integers(0, 2, scrambled[:, 120:].shape)
no_future_dependency = build_snapshot(scrambled, churn11, 120)[0].equals(X11_te)
### END CODE HERE ###
print(f"train: {len(X11_tr):,} dòng (3 ngày chụp) | test (ngày 120): {len(X11_te):,} dòng, tỉ lệ rời bỏ {y11_te.mean():.1%} | AUC theo thời gian = {auc11:.3f}")
print("đặc trưng không phụ thuộc tương lai:", no_future_dependency)

In [ ]:
X_chk, y_chk = build_snapshot(activity11, churn11, 90)
check_equal(len(X_chk), int((churn11 > 90).sum()), "chỉ khách còn hoạt động tại ngày chụp")
check_equal(list(X_chk.columns), ["active_last_7", "active_last_30", "active_total", "days_since_last", "trend"], "đủ 5 đặc trưng")
check_equal(int(y_chk.sum()), int(((churn11 > 90) & (churn11 <= 120)).sum()), "nhãn = rời bỏ trong 30 ngày sau ngày chụp")
check_true(no_future_dependency, "kiểm thử tính chất: dữ liệu tương lai không ảnh hưởng đặc trưng")
probe = activity11.copy()
probe[:, 90:] = 1 - probe[:, 90:]
check_true(build_snapshot(probe, churn11, 90)[0].equals(X_chk), "kiểm tra độc lập: lật mọi ngày ≥ 90 không đổi đặc trưng tại ngày 90")
check_greater(auc11, 0.75, "mô hình đúng thời điểm vẫn dự đoán tốt trên ngày chụp tương lai")

**Nhận xét:** Ghép nhiều ngày chụp làm tăng dữ liệu huấn luyện và dạy mô hình hành vi ở nhiều giai đoạn; một khách xuất hiện ở nhiều ngày chụp — vì vậy **không** được chia train/test ngẫu nhiên theo dòng (cùng khách ở cả hai phía) mà chia **theo thời gian**. Lưu ý khoảng cách: nhãn của ngày chụp train phải **đã biết** tại thời điểm test (90 + 30 ≤ 120) — nếu dùng ngày chụp 100 làm train, nhãn của nó (ngày 101–130) chồng lấn với thời gian test, một dạng rò rỉ tinh vi. Kiểm thử tính chất ("thay dữ liệu tương lai, đặc trưng không đổi") là cách **tự động hoá** việc chống rò rỉ thời điểm — nên đặt trong bộ test của pipeline đặc trưng, vì lỗi rò rỉ không bao giờ tự báo lỗi.